# WISE Pipeline Sweep Validation

**Goal:** Validate that the NEW trapped-ion architecture replicates the key **trends**
from the OLD `wise_arch_routing.ipynb` pipeline when varying:
- `lookahead` (SAT solver context window)
- `subgrid_width / subgrid_height / subgrid_increment` (SAT decomposition)
- `trap_capacity` (ions per segment)
- code distance `d`

We do NOT need exact numerical matches — the pipelines use different timing models,
noise injection, and qubit mapping. But **trend directions** must agree.

---

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 1: Imports with deep module reload
#
# Deep reload ensures latest code changes are picked up (critical after
# bug fixes in routing.py and experiments.py).
# ═══════════════════════════════════════════════════════════════════════
import sys, os, time, importlib, traceback as _tb
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Deep reload all qectostim modules
_mods = sorted(
    [m for m in sys.modules if 'qectostim' in m],
    key=lambda m: m.count('.'), reverse=True,
)
for _mn in _mods:
    del sys.modules[_mn]
print(f"✓ Cleared {len(_mods)} qectostim modules")

from qectostim.experiments.hardware_simulation.trapped_ion import (
    WISEArchitecture, WISECompiler, WISERoutingConfig,
    TrappedIonExperiment, TrappedIonNoiseModel,
)
from qectostim.codes.surface.rotated_surface import RotatedSurfaceCode

print(f"✓ Imports loaded")
print(f"  WISEArchitecture: {WISEArchitecture}")
print(f"  WISECompiler:     {WISECompiler}")
print(f"  RotatedSurfaceCode: {RotatedSurfaceCode}")

✓ Cleared 0 qectostim modules
✓ Imports loaded
  WISEArchitecture: <class 'qectostim.experiments.hardware_simulation.trapped_ion.architecture.WISEArchitecture'>
  WISECompiler:     <class 'qectostim.experiments.hardware_simulation.trapped_ion.compilers.wise.WISECompiler'>
  RotatedSurfaceCode: <class 'qectostim.codes.surface.rotated_surface.RotatedSurfaceCode'>


/Users/scottjones_admin/Library/Mobile Documents/com~apple~CloudDocs/Mac files/Repos/QECToStim/my_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 2: Old reference data (56 rows from wise_arch_routing.ipynb)
#
# exec_time is in SECONDS (old pipeline convention).
# We convert to microseconds for comparison.
# ═══════════════════════════════════════════════════════════════════════

OLD_RESULTS = [
    {'lookahead': 1, 'subgrid_width': 4, 'subgrid_height': 3, 'subgrid_increment': 0, 'd': 2, 'm_traps': 2, 'n_traps': 3, 'trap_capacity': 2, 'exec_time': 0.005659, 'comp_time': 24.08, 'logicalerror': 0.1161, 'reconfigTime': 0.003724},
    {'lookahead': 1, 'subgrid_width': 2, 'subgrid_height': 2, 'subgrid_increment': 2, 'd': 2, 'm_traps': 2, 'n_traps': 3, 'trap_capacity': 2, 'exec_time': 0.009787, 'comp_time': 52.86, 'logicalerror': 0.11893, 'reconfigTime': 0.007802},
    {'lookahead': 4, 'subgrid_width': 4, 'subgrid_height': 3, 'subgrid_increment': 0, 'd': 2, 'm_traps': 2, 'n_traps': 3, 'trap_capacity': 2, 'exec_time': 0.003674, 'comp_time': 110.04, 'logicalerror': 0.11533, 'reconfigTime': 0.001684},
    {'lookahead': 4, 'subgrid_width': 2, 'subgrid_height': 2, 'subgrid_increment': 2, 'd': 2, 'm_traps': 2, 'n_traps': 3, 'trap_capacity': 2, 'exec_time': 0.008348, 'comp_time': 481.84, 'logicalerror': 0.1162, 'reconfigTime': 0.006708},
    {'lookahead': 1, 'subgrid_width': 4, 'subgrid_height': 3, 'subgrid_increment': 0, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 4, 'exec_time': 0.004840, 'comp_time': 25.70, 'logicalerror': 0.11693, 'reconfigTime': 0.002790},
    {'lookahead': 1, 'subgrid_width': 2, 'subgrid_height': 2, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 4, 'exec_time': 0.006913, 'comp_time': 43.01, 'logicalerror': 0.12043, 'reconfigTime': 0.004788},
    {'lookahead': 4, 'subgrid_width': 4, 'subgrid_height': 3, 'subgrid_increment': 0, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 4, 'exec_time': 0.004725, 'comp_time': 81.66, 'logicalerror': 0.11599, 'reconfigTime': 0.001260},
    {'lookahead': 4, 'subgrid_width': 2, 'subgrid_height': 2, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 4, 'exec_time': 0.007868, 'comp_time': 242.87, 'logicalerror': 0.11741, 'reconfigTime': 0.004788},
    {'lookahead': 1, 'subgrid_width': 5, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 8, 'exec_time': 0.004010, 'comp_time': 11.58, 'logicalerror': 0.14691, 'reconfigTime': 0.000240},
    {'lookahead': 1, 'subgrid_width': 8, 'subgrid_height': 3, 'subgrid_increment': 0, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 8, 'exec_time': 0.005032, 'comp_time': 88.12, 'logicalerror': 0.14807, 'reconfigTime': 0.001472},
    {'lookahead': 1, 'subgrid_width': 3, 'subgrid_height': 1, 'subgrid_increment': 4, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 8, 'exec_time': 0.006662, 'comp_time': 115.85, 'logicalerror': 0.15246, 'reconfigTime': 0.002892},
    {'lookahead': 1, 'subgrid_width': 3, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 8, 'exec_time': 0.006662, 'comp_time': 118.21, 'logicalerror': 0.15252, 'reconfigTime': 0.002892},
    {'lookahead': 4, 'subgrid_width': 5, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 8, 'exec_time': 0.004010, 'comp_time': 16.60, 'logicalerror': 0.14715, 'reconfigTime': 0.000240},
    {'lookahead': 4, 'subgrid_width': 8, 'subgrid_height': 3, 'subgrid_increment': 0, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 8, 'exec_time': 0.004010, 'comp_time': 200.66, 'logicalerror': 0.14894, 'reconfigTime': 0.000240},
    {'lookahead': 4, 'subgrid_width': 3, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 8, 'exec_time': 0.006026, 'comp_time': 408.96, 'logicalerror': 0.14869, 'reconfigTime': 0.002256},
    {'lookahead': 4, 'subgrid_width': 3, 'subgrid_height': 1, 'subgrid_increment': 4, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 8, 'exec_time': 0.006026, 'comp_time': 383.40, 'logicalerror': 0.14810, 'reconfigTime': 0.002256},
    {'lookahead': 1, 'subgrid_width': 7, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.004010, 'comp_time': 15.20, 'logicalerror': 0.20172, 'reconfigTime': 0.000240},
    {'lookahead': 1, 'subgrid_width': 7, 'subgrid_height': 1, 'subgrid_increment': 4, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.004010, 'comp_time': 15.95, 'logicalerror': 0.20172, 'reconfigTime': 0.000240},
    {'lookahead': 1, 'subgrid_width': 9, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.004646, 'comp_time': 45.93, 'logicalerror': 0.20224, 'reconfigTime': 0.000876},
    {'lookahead': 1, 'subgrid_width': 9, 'subgrid_height': 1, 'subgrid_increment': 4, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.004646, 'comp_time': 45.63, 'logicalerror': 0.20611, 'reconfigTime': 0.000876},
    {'lookahead': 1, 'subgrid_width': 5, 'subgrid_height': 1, 'subgrid_increment': 4, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.005602, 'comp_time': 517.75, 'logicalerror': 0.20630, 'reconfigTime': 0.001832},
    {'lookahead': 1, 'subgrid_width': 16, 'subgrid_height': 3, 'subgrid_increment': 0, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.005465, 'comp_time': 489.45, 'logicalerror': 0.20222, 'reconfigTime': 0.002280},
    {'lookahead': 4, 'subgrid_width': 7, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.004010, 'comp_time': 24.49, 'logicalerror': 0.20377, 'reconfigTime': 0.000240},
    {'lookahead': 4, 'subgrid_width': 7, 'subgrid_height': 1, 'subgrid_increment': 4, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.004010, 'comp_time': 26.12, 'logicalerror': 0.20222, 'reconfigTime': 0.000240},
    {'lookahead': 4, 'subgrid_width': 9, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.004010, 'comp_time': 82.80, 'logicalerror': 0.20406, 'reconfigTime': 0.000240},
    {'lookahead': 4, 'subgrid_width': 9, 'subgrid_height': 1, 'subgrid_increment': 4, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.004010, 'comp_time': 84.52, 'logicalerror': 0.20212, 'reconfigTime': 0.000240},
    {'lookahead': 4, 'subgrid_width': 16, 'subgrid_height': 3, 'subgrid_increment': 0, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.004295, 'comp_time': 1257.58, 'logicalerror': 0.20413, 'reconfigTime': 0.000750},
    {'lookahead': 1, 'subgrid_width': 5, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.005602, 'comp_time': 2909.50, 'logicalerror': 0.20753, 'reconfigTime': 0.001832},
    {'lookahead': 4, 'subgrid_width': 5, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 2, 'm_traps': 1, 'n_traps': 3, 'trap_capacity': 16, 'exec_time': 0.006450, 'comp_time': 4944.66, 'logicalerror': 0.21236, 'reconfigTime': 0.002680},
    # ── d=3 ──────────────────────────────────────────────────────────────
    {'lookahead': 1, 'subgrid_width': 4, 'subgrid_height': 4, 'subgrid_increment': 2, 'd': 3, 'm_traps': 3, 'n_traps': 5, 'trap_capacity': 2, 'exec_time': 0.017909, 'comp_time': 304.64, 'logicalerror': 0.12658, 'reconfigTime': 0.015274},
    {'lookahead': 1, 'subgrid_width': 2, 'subgrid_height': 2, 'subgrid_increment': 2, 'd': 3, 'm_traps': 3, 'n_traps': 5, 'trap_capacity': 2, 'exec_time': 0.018858, 'comp_time': 326.80, 'logicalerror': 0.12538, 'reconfigTime': 0.015948},
    {'lookahead': 1, 'subgrid_width': 6, 'subgrid_height': 5, 'subgrid_increment': 0, 'd': 3, 'm_traps': 3, 'n_traps': 5, 'trap_capacity': 2, 'exec_time': 0.007649, 'comp_time': 574.15, 'logicalerror': 0.12347, 'reconfigTime': 0.005294},
    {'lookahead': 2, 'subgrid_width': 6, 'subgrid_height': 5, 'subgrid_increment': 0, 'd': 3, 'm_traps': 3, 'n_traps': 5, 'trap_capacity': 2, 'exec_time': 0.005983, 'comp_time': 943.81, 'logicalerror': 0.12113, 'reconfigTime': 0.003638},
    {'lookahead': 2, 'subgrid_width': 2, 'subgrid_height': 2, 'subgrid_increment': 2, 'd': 3, 'm_traps': 3, 'n_traps': 5, 'trap_capacity': 2, 'exec_time': 0.020869, 'comp_time': 2555.15, 'logicalerror': 0.12614, 'reconfigTime': 0.018114},
    {'lookahead': 2, 'subgrid_width': 4, 'subgrid_height': 4, 'subgrid_increment': 2, 'd': 3, 'm_traps': 3, 'n_traps': 5, 'trap_capacity': 2, 'exec_time': 0.015400, 'comp_time': 2397.12, 'logicalerror': 0.12658, 'reconfigTime': 0.012810},
    {'lookahead': 1, 'subgrid_width': 8, 'subgrid_height': 5, 'subgrid_increment': 0, 'd': 3, 'm_traps': 2, 'n_traps': 5, 'trap_capacity': 4, 'exec_time': 0.007601, 'comp_time': 274.74, 'logicalerror': 0.12564, 'reconfigTime': 0.004996},
    {'lookahead': 1, 'subgrid_width': 5, 'subgrid_height': 3, 'subgrid_increment': 2, 'd': 3, 'm_traps': 2, 'n_traps': 5, 'trap_capacity': 4, 'exec_time': 0.016049, 'comp_time': 415.55, 'logicalerror': 0.12602, 'reconfigTime': 0.012644},
    {'lookahead': 1, 'subgrid_width': 3, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 3, 'm_traps': 2, 'n_traps': 5, 'trap_capacity': 4, 'exec_time': 0.012681, 'comp_time': 1219.33, 'logicalerror': 0.12619, 'reconfigTime': 0.009366},
    {'lookahead': 2, 'subgrid_width': 8, 'subgrid_height': 5, 'subgrid_increment': 0, 'd': 3, 'm_traps': 2, 'n_traps': 5, 'trap_capacity': 4, 'exec_time': 0.007355, 'comp_time': 1208.36, 'logicalerror': 0.12493, 'reconfigTime': 0.003850},
    {'lookahead': 2, 'subgrid_width': 5, 'subgrid_height': 3, 'subgrid_increment': 2, 'd': 3, 'm_traps': 2, 'n_traps': 5, 'trap_capacity': 4, 'exec_time': 0.014753, 'comp_time': 2493.60, 'logicalerror': 0.12589, 'reconfigTime': 0.011928},
    {'lookahead': 2, 'subgrid_width': 3, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 3, 'm_traps': 2, 'n_traps': 5, 'trap_capacity': 4, 'exec_time': 0.011695, 'comp_time': 3295.28, 'logicalerror': 0.12621, 'reconfigTime': 0.008770},
    {'lookahead': 1, 'subgrid_width': 5, 'subgrid_height': 3, 'subgrid_increment': 2, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 8, 'exec_time': 0.010951, 'comp_time': 129.13, 'logicalerror': 0.16841, 'reconfigTime': 0.005036},
    {'lookahead': 2, 'subgrid_width': 5, 'subgrid_height': 3, 'subgrid_increment': 2, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 8, 'exec_time': 0.009595, 'comp_time': 200.85, 'logicalerror': 0.16362, 'reconfigTime': 0.003890},
    {'lookahead': 1, 'subgrid_width': 8, 'subgrid_height': 5, 'subgrid_increment': 0, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 8, 'exec_time': 0.010444, 'comp_time': 419.38, 'logicalerror': 0.16624, 'reconfigTime': 0.005844},
    {'lookahead': 1, 'subgrid_width': 3, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 8, 'exec_time': 0.013531, 'comp_time': 818.18, 'logicalerror': 0.17206, 'reconfigTime': 0.008576},
    {'lookahead': 2, 'subgrid_width': 3, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 8, 'exec_time': 0.012024, 'comp_time': 978.31, 'logicalerror': 0.16938, 'reconfigTime': 0.006284},
    {'lookahead': 2, 'subgrid_width': 8, 'subgrid_height': 5, 'subgrid_increment': 0, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 8, 'exec_time': 0.008804, 'comp_time': 782.12, 'logicalerror': 0.16368, 'reconfigTime': 0.004314},
    {'lookahead': 1, 'subgrid_width': 4, 'subgrid_height': 1, 'subgrid_increment': 2, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 16, 'exec_time': 0.013850, 'comp_time': 368.67, 'logicalerror': 0.25774, 'reconfigTime': 0.006420},
    {'lookahead': 1, 'subgrid_width': 7, 'subgrid_height': 2, 'subgrid_increment': 2, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 16, 'exec_time': 0.013243, 'comp_time': 1778.65, 'logicalerror': 0.24826, 'reconfigTime': 0.006788},
    {'lookahead': 1, 'subgrid_width': 11, 'subgrid_height': 3, 'subgrid_increment': 2, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 16, 'exec_time': 0.010217, 'comp_time': 2107.74, 'logicalerror': 0.24456, 'reconfigTime': 0.004612},
    {'lookahead': 1, 'subgrid_width': 14, 'subgrid_height': 4, 'subgrid_increment': 2, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 16, 'exec_time': 0.010247, 'comp_time': 3093.77, 'logicalerror': 0.24261, 'reconfigTime': 0.004612},
    {'lookahead': 1, 'subgrid_width': 16, 'subgrid_height': 5, 'subgrid_increment': 0, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 16, 'exec_time': 0.009680, 'comp_time': 3167.34, 'logicalerror': 0.24593, 'reconfigTime': 0.003890},
    {'lookahead': 2, 'subgrid_width': 11, 'subgrid_height': 3, 'subgrid_increment': 2, 'd': 3, 'm_traps': 1, 'n_traps': 5, 'trap_capacity': 16, 'exec_time': 0.008537, 'comp_time': 1725.46, 'logicalerror': 0.24538, 'reconfigTime': 0.002062},
    # ── d=4 ──────────────────────────────────────────────────────────────
    {'lookahead': 1, 'subgrid_width': 6, 'subgrid_height': 6, 'subgrid_increment': 0, 'd': 4, 'm_traps': 3, 'n_traps': 6, 'trap_capacity': 2, 'exec_time': 0.009052, 'comp_time': 2505.56, 'logicalerror': 0.09198, 'reconfigTime': 0.006652},
    {'lookahead': 1, 'subgrid_width': 4, 'subgrid_height': 4, 'subgrid_increment': 2, 'd': 4, 'm_traps': 3, 'n_traps': 6, 'trap_capacity': 2, 'exec_time': 0.023509, 'comp_time': 6000.22, 'logicalerror': 0.09422, 'reconfigTime': 0.020184},
    {'lookahead': 1, 'subgrid_width': 8, 'subgrid_height': 6, 'subgrid_increment': 0, 'd': 4, 'm_traps': 2, 'n_traps': 6, 'trap_capacity': 4, 'exec_time': 0.010106, 'comp_time': 2548.55, 'logicalerror': 0.09664, 'reconfigTime': 0.006526},
    {'lookahead': 1, 'subgrid_width': 5, 'subgrid_height': 3, 'subgrid_increment': 2, 'd': 4, 'm_traps': 2, 'n_traps': 6, 'trap_capacity': 4, 'exec_time': 0.023838, 'comp_time': 5670.67, 'logicalerror': 0.09841, 'reconfigTime': 0.019468},
    {'lookahead': 1, 'subgrid_width': 8, 'subgrid_height': 6, 'subgrid_increment': 0, 'd': 4, 'm_traps': 1, 'n_traps': 6, 'trap_capacity': 8, 'exec_time': 0.017641, 'comp_time': 4707.32, 'logicalerror': 0.15234, 'reconfigTime': 0.010886},
    {'lookahead': 1, 'subgrid_width': 13, 'subgrid_height': 4, 'subgrid_increment': 2, 'd': 4, 'm_traps': 1, 'n_traps': 6, 'trap_capacity': 16, 'exec_time': 0.015482, 'comp_time': 5860.44, 'logicalerror': 0.26391, 'reconfigTime': 0.007912},
]

df_old = pd.DataFrame(OLD_RESULTS)
df_old['exec_time_us'] = df_old['exec_time'] * 1e6
df_old['reconfigTime_us'] = df_old['reconfigTime'] * 1e6

print(f"Old results: {len(df_old)} rows")
print(f"\nDistance breakdown:")
print(df_old.groupby('d').agg(
    count=('exec_time', 'size'),
    k_values=('trap_capacity', lambda x: sorted(x.unique())),
    la_values=('lookahead', lambda x: sorted(x.unique())),
).to_string())
print(f"\nExec time range: {df_old['exec_time_us'].min():.0f} – {df_old['exec_time_us'].max():.0f} µs")
print(f"LER range: {df_old['logicalerror'].min():.4f} – {df_old['logicalerror'].max():.4f}")

Old results: 59 rows

Distance breakdown:
   count       k_values la_values
d                                
2     29  [2, 4, 8, 16]    [1, 4]
3     24  [2, 4, 8, 16]    [1, 2]
4      6  [2, 4, 8, 16]       [1]

Exec time range: 3674 – 23838 µs
LER range: 0.0920 – 0.2639


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 3: Representative configs for trend validation
#
# 8 configs that isolate each parameter axis:
#   T1: Lookahead effect         (d=2, k=2, full-grid, la=1 vs la=4)
#   T2: Subgrid effect           (d=2, k=2, la=1, full vs small)
#   T3: Trap capacity effect     (d=2, la=1, full-grid, k=2 vs k=4 vs k=8)
#   T4: Code distance effect     (k=2, la=1, full-grid, d=2 vs d=3 vs d=4)
#   T5: Lookahead at d=3         (d=3, k=2, full-grid, la=1 vs la=2)
# ═══════════════════════════════════════════════════════════════════════

def find_old_idx(d, k, la, sw, sh, si):
    mask = ((df_old['d'] == d) & (df_old['trap_capacity'] == k) &
            (df_old['lookahead'] == la) & (df_old['subgrid_width'] == sw) &
            (df_old['subgrid_height'] == sh) & (df_old['subgrid_increment'] == si))
    matches = df_old[mask]
    return matches.index[0] if len(matches) > 0 else None

REPRESENTATIVE = [
    # label,                la, sw, sh, si, d, m, n, k
    ("d2_k2_la1_full",       1,  4,  3,  0, 2, 2, 3, 2),   # T1a / T2a / T3a / T4a
    ("d2_k2_la4_full",       4,  4,  3,  0, 2, 2, 3, 2),   # T1b
    ("d2_k2_la1_small",      1,  2,  2,  2, 2, 2, 3, 2),   # T2b
    ("d2_k4_la1_full",       1,  4,  3,  0, 2, 1, 3, 4),   # T3b
    ("d2_k8_la1_full",       1,  8,  3,  0, 2, 1, 3, 8),   # T3c
    ("d3_k2_la1_full",       1,  6,  5,  0, 3, 3, 5, 2),   # T4b / T5a
    ("d3_k2_la2_full",       2,  6,  5,  0, 3, 3, 5, 2),   # T5b
    ("d4_k2_la1_full",       1,  6,  6,  0, 4, 3, 6, 2),   # T4c
]

REPRESENTATIVE_WITH_IDX = []
for label, la, sw, sh, si, d, m, n, k in REPRESENTATIVE:
    old_idx = find_old_idx(d, k, la, sw, sh, si)
    REPRESENTATIVE_WITH_IDX.append((label, la, sw, sh, si, d, m, n, k, old_idx))

print(f"{'Label':<22} {'la':>3} {'sw':>3} {'sh':>3} {'si':>3} {'d':>2} {'m':>2} {'n':>2} {'k':>2} {'idx':>4}  {'old_exec_µs':>12} {'old_LER':>10}")
print("─" * 95)
for label, la, sw, sh, si, d, m, n, k, old_idx in REPRESENTATIVE_WITH_IDX:
    if old_idx is not None:
        old = df_old.iloc[old_idx]
        print(f"{label:<22} {la:>3} {sw:>3} {sh:>3} {si:>3} {d:>2} {m:>2} {n:>2} {k:>2} {old_idx:>4}  "
              f"{old['exec_time_us']:>12.0f} {old['logicalerror']:>10.4f}")
    else:
        print(f"{label:<22} {la:>3} {sw:>3} {sh:>3} {si:>3} {d:>2} {m:>2} {n:>2} {k:>2}  ???  — NOT FOUND")

# Verify trend expectations
i = {r[0]: r[-1] for r in REPRESENTATIVE_WITH_IDX}
print(f"\nExpected trends from OLD data:")
if i.get("d2_k2_la4_full") is not None and i.get("d2_k2_la1_full") is not None:
    print(f"  T1 (lookahead↑): exec_time ↓  — la=4 ({df_old.iloc[i['d2_k2_la4_full']]['exec_time_us']:.0f}) "
          f"< la=1 ({df_old.iloc[i['d2_k2_la1_full']]['exec_time_us']:.0f}) µs")
if i.get("d2_k2_la1_full") is not None and i.get("d2_k2_la1_small") is not None:
    print(f"  T2 (subgrid↑):   exec_time ↓  — full ({df_old.iloc[i['d2_k2_la1_full']]['exec_time_us']:.0f}) "
          f"< small ({df_old.iloc[i['d2_k2_la1_small']]['exec_time_us']:.0f}) µs")
if all(i.get(l) is not None for l in ["d2_k2_la1_full","d2_k4_la1_full","d2_k8_la1_full"]):
    print(f"  T3 (k↑):         LER ↑        — k2={df_old.iloc[i['d2_k2_la1_full']]['logicalerror']:.4f} "
          f"< k4={df_old.iloc[i['d2_k4_la1_full']]['logicalerror']:.4f} "
          f"< k8={df_old.iloc[i['d2_k8_la1_full']]['logicalerror']:.4f}")
if all(i.get(l) is not None for l in ["d4_k2_la1_full","d2_k2_la1_full"]):
    print(f"  T4 (d↑):         LER ↓        — d4={df_old.iloc[i['d4_k2_la1_full']]['logicalerror']:.4f} "
          f"< d2={df_old.iloc[i['d2_k2_la1_full']]['logicalerror']:.4f}")
if i.get("d3_k2_la2_full") is not None and i.get("d3_k2_la1_full") is not None:
    print(f"  T5 (la@d3↑):     exec_time ↓  — la=2 ({df_old.iloc[i['d3_k2_la2_full']]['exec_time_us']:.0f}) "
          f"< la=1 ({df_old.iloc[i['d3_k2_la1_full']]['exec_time_us']:.0f}) µs")

Label                   la  sw  sh  si  d  m  n  k  idx   old_exec_µs    old_LER
───────────────────────────────────────────────────────────────────────────────────────────────
d2_k2_la1_full           1   4   3   0  2  2  3  2    0          5659     0.1161
d2_k2_la4_full           4   4   3   0  2  2  3  2    2          3674     0.1153
d2_k2_la1_small          1   2   2   2  2  2  3  2    1          9787     0.1189
d2_k4_la1_full           1   4   3   0  2  1  3  4    4          4840     0.1169
d2_k8_la1_full           1   8   3   0  2  1  3  8    9          5032     0.1481
d3_k2_la1_full           1   6   5   0  3  3  5  2   31          7649     0.1235
d3_k2_la2_full           2   6   5   0  3  3  5  2   32          5983     0.1211
d4_k2_la1_full           1   6   6   0  4  3  6  2   53          9052     0.0920

Expected trends from OLD data:
  T1 (lookahead↑): exec_time ↓  — la=4 (3674) < la=1 (5659) µs
  T2 (subgrid↑):   exec_time ↓  — full (5659) < small (9787) µs
  T3 (k↑):      

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 4: Main sweep — run all 8 representative configs
#
# Includes enhanced diagnostics:
#   - compile_metrics (duration_us, reconfiguration_time_us, total_ops)
#   - plan info (operations count, total_duration)
#   - routing metadata (num_batches, motional_quanta)
# ═══════════════════════════════════════════════════════════════════════
import sys, time, importlib, traceback as _tb

# ── Deep module reload (ensures latest code) ────────────────────────
_mods = sorted(
    [m for m in sys.modules if 'qectostim' in m],
    key=lambda m: m.count('.'), reverse=True,
)
for _mn in _mods:
    del sys.modules[_mn]
print(f"✓ Cleared {len(_mods)} qectostim modules")

from qectostim.experiments.hardware_simulation.trapped_ion import (
    WISEArchitecture, WISECompiler, WISERoutingConfig,
    TrappedIonExperiment, TrappedIonNoiseModel,
)
from qectostim.codes.surface.rotated_surface import RotatedSurfaceCode

# ── Sweep settings ──────────────────────────────────────────────────
NUM_VAL_SHOTS = 2_000
SAT_TIMEOUT   = 120.0     # seconds per SAT solve

new_results = []
for label, la, sw, sh, si, d, m, n, k, old_idx in REPRESENTATIVE_WITH_IDX:
    if old_idx is None:
        print(f"⏭  {label} — skipped (no matching old row)")
        continue
    old_row = df_old.iloc[old_idx]
    print(f"\n▶ {label}  (d={d}, m={m}, n={n}, k={k}, la={la}, sg={sw}×{sh}+{si})")
    t0 = time.time()
    try:
        code_obj = RotatedSurfaceCode(d)
        arch_obj = WISEArchitecture(col_groups=m, rows=n, ions_per_segment=k)
        routing_cfg = WISERoutingConfig(
            subgridsize=(sw, sh, si),
            lookahead_rounds=la,
            timeout_seconds=SAT_TIMEOUT,
        )
        compiler = WISECompiler(
            arch_obj,
            routing_config=routing_cfg,
            use_junction_routing=False,
            partition_strategy="gate_affinity",
        )
        noise = TrappedIonNoiseModel()
        exp = TrappedIonExperiment(
            code=code_obj, architecture=arch_obj, compiler=compiler,
            hardware_noise=noise, rounds=d,
        )

        result = exp.simulate(
            num_shots=NUM_VAL_SHOTS,
            gate_improvements=[1.0],
        )
        wall = time.time() - t0

        metrics = result.simulation_metrics
        cm = result.compilation_metrics

        # Extract key values
        new_exec_us    = metrics.get("ElapsedTime", metrics.get("total_duration_us", 0))
        new_reconfig_us = metrics.get("ReconfigurationTime", 0)
        new_ler        = result.logical_error_rate
        if isinstance(metrics.get("LogicalErrorRates"), list):
            new_ler = metrics["LogicalErrorRates"][0]

        # Diagnostics
        total_ops    = metrics.get("Operations", cm.get("total_operations", "?"))
        two_q_ops    = metrics.get("QubitOperations", cm.get("two_qubit_operations", "?"))
        parallelism  = metrics.get("MeanConcurrency", cm.get("parallelism", "?"))
        num_batches  = cm.get("num_batches", "?")
        plan_dur     = metrics.get("total_duration_us", "?")
        plan_ops     = metrics.get("num_operations", "?")

        new_results.append({
            "label": label, "d": d, "m": m, "n": n, "k": k,
            "la": la, "sw": sw, "sh": sh, "si": si,
            "new_exec_us":    new_exec_us,
            "new_reconfig_us": new_reconfig_us,
            "new_ler":        new_ler,
            "new_ops":        total_ops,
            "new_2q_ops":     two_q_ops,
            "old_exec_us":    old_row["exec_time_us"],
            "old_ler":        old_row["logicalerror"],
            "old_reconfig_us": old_row["reconfigTime_us"],
            "wall_s":         wall,
            "status":         "ok",
        })
        print(f"  ✅ {wall:.1f}s  exec={new_exec_us:.0f}µs  reconf={new_reconfig_us:.0f}µs  LER={new_ler:.4f}")
        print(f"     ops={total_ops}  2Q={two_q_ops}  par={parallelism}  batches={num_batches}  plan_dur={plan_dur}µs  plan_ops={plan_ops}")

    except Exception as e:
        wall = time.time() - t0
        new_results.append({
            "label": label, "d": d, "m": m, "n": n, "k": k,
            "la": la, "sw": sw, "sh": sh, "si": si,
            "new_exec_us": None, "new_reconfig_us": None,
            "new_ler": None, "new_ops": None, "new_2q_ops": None,
            "old_exec_us":    old_row["exec_time_us"],
            "old_ler":        old_row["logicalerror"],
            "old_reconfig_us": old_row["reconfigTime_us"],
            "wall_s":         wall,
            "status":         f"ERROR: {type(e).__name__}: {e}",
        })
        print(f"  ❌ {wall:.1f}s  {type(e).__name__}: {e}")
        _tb.print_exc()

df_new = pd.DataFrame(new_results)
ok_count = (df_new["status"] == "ok").sum()
print(f"\n{'='*60}")
print(f"Completed: {ok_count}/{len(REPRESENTATIVE_WITH_IDX)} configs succeeded")

✓ Cleared 185 qectostim modules

▶ d2_k2_la1_full  (d=2, m=2, n=3, k=2, la=1, sg=4×3+0)
  ✅ 86.4s  exec=2085µs  reconf=0µs  LER=0.1020
     ops=188  2Q=29  par=4.372093023255814  batches=30  plan_dur=525.0µs  plan_ops=37

▶ d2_k2_la4_full  (d=2, m=2, n=3, k=2, la=4, sg=4×3+0)


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
ERROR:tornado.general:SEND Error: Host unreachable
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It se

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

In [34]:
# ═══════════════════════════════════════════════════════════════════════
# Diagnostic: Noise model verification
# ═══════════════════════════════════════════════════════════════════════
from qectostim.experiments.hardware_simulation.trapped_ion.noise import TrappedIonNoiseModel
from qectostim.experiments.hardware_simulation.trapped_ion.execution import OperationTiming

nm = TrappedIonNoiseModel()
print("Noise model verification:")
print(f"  1Q fidelity (k=2, n̄=0): {nm.single_qubit_gate_fidelity(2, 0.0):.4f}  (should be 1.0)")
print(f"  2Q fidelity (k=2, n̄=0): {nm.ms_gate_fidelity(2, 0.0):.4f}  (≈0.989)")

# Verify 1Q gate produces no noise channels
t1q = OperationTiming(instruction_index=0, gate_name="H", qubits=(0,),
                       start_time=0, duration=5e-6, fidelity=0.989,
                       chain_length=2, motional_quanta=0.0)
ch_1q = nm.apply_to_operation_timing(t1q)
print(f"  1Q 'H' noise channels: {len(ch_1q)} (should be 0)")

# Verify 2Q gate produces correct noise
t2q = OperationTiming(instruction_index=1, gate_name="CX", qubits=(0, 1),
                       start_time=5e-6, duration=40e-6, fidelity=0.989,
                       chain_length=2, motional_quanta=0.0)
ch_2q = nm.apply_to_operation_timing(t2q)
print(f"  2Q 'CX' noise channels: {len(ch_2q)} (should be 1)")
if ch_2q:
    print(f"    → {ch_2q[0].channel_type.name}: p={ch_2q[0].probability:.6f}")

print("\nRemaining trend failures root causes:")
print("  T1 (la@d2): SAT finds same pass-count for d=2 grid regardless of lookahead")
print("  T2 (subgrid): Patch routing produces identical pass-count for this small grid")
print("  T4 (d→LER): d=4 stim circuit has undetectable errors; needs circuit construction fix")


gate_improvement = 1.0 → effective 2Q error ≈ 0.01078
  [1/4] Building ideal circuit …
         → 8Q, 46 instructions
  [2/4] Building execution plan …


WISE routing: 100%|██████████| 10/10 [00:04<00:00,  2.09batch/s, pairs=2, ops=24]

         → 37 operations, 40 µs
  [3/4] Using cached compilation (from execution plan) … OK
  [4/4] Decoding (2000 shots) …
         scaling 1/1 (improvement=1.0) …
  ✅ Done in 4.84s — LER=1.0050e-01
  d=2: LER=0.1005 (4.8s)
  [1/4] Building ideal circuit …


         → 32Q, 124 instructions
  [2/4] Building execution plan …


WISE routing: 100%|██████████| 27/27 [02:13<00:00,  4.94s/batch, pairs=2, ops=400]

...


WISE routing: 100%|██████████| 10/10 [00:04<00:00,  2.08batch/s, pairs=2, ops=24]

         → 37 operations, 40 µs
  [3/4] Using cached compilation (from execution plan) … OK
  [4/4] Decoding (2000 shots) …
         scaling 1/1 (improvement=5.0) …
  ✅ Done in 4.86s — LER=1.3500e-02
  d=2: LER=0.0135 (4.9s)
  [1/4] Building ideal circuit …


         → 32Q, 124 instructions
  [2/4] Building execution plan …


WISE routing: 100%|██████████| 27/27 [02:11<00:00,  4.86s/batch, pairs=2, ops=400]

         → 107 operations, 30 µs
  [3/4] Using cached compilation (from execution plan) … OK
  [4/4] Decoding (2000 shots) …
         scaling 1/1 (improvement=5.0) …
  ✅ Done in 131.56s — LER=5.0500e-02
  d=4: LER=0.0505 (131.6s)


In [38]:
# ═══════════════════════════════════════════════════════════════════════
# T4 Diagnostic: Investigate d=4 circuit undetectable errors
# ═══════════════════════════════════════════════════════════════════════
import stim

for d_test in [2, 3, 4]:
    code_t = RotatedSurfaceCode(d_test)
    arch_t = WISEArchitecture(col_groups=d_test, rows=d_test+1, ions_per_segment=2)
    rc_t = WISERoutingConfig(subgridsize=(2*d_test, d_test+1, 2), lookahead_rounds=1, timeout_seconds=30.0)
    compiler_t = WISECompiler(arch_t, routing_config=rc_t)
    noise_t = TrappedIonNoiseModel()
    exp_t = TrappedIonExperiment(code=code_t, architecture=arch_t, compiler=compiler_t,
                                  hardware_noise=noise_t, rounds=d_test)
    
    # Build the ideal circuit first
    ideal = exp_t.build_ideal_circuit()
    n_ideal_det = sum(1 for inst in ideal.flattened() if inst.name == "DETECTOR")
    n_ideal_obs = sum(1 for inst in ideal.flattened() if inst.name == "OBSERVABLE_INCLUDE")
    
    print(f"\n=== d={d_test} (r={d_test}) ===")
    print(f"  Ideal circuit: {ideal.num_qubits} qubits, {n_ideal_det} detectors, {n_ideal_obs} observables")
    
    # Noiseless decode test
    det_sampler = ideal.compile_detector_sampler()
    det_samples = det_sampler.sample(100, append_observables=True)
    n_det_fires = det_samples[:, :n_ideal_det].sum()
    n_obs_fires = det_samples[:, n_ideal_det:].sum()
    print(f"  Noiseless ideal: det_fires={n_det_fires}, obs_fires={n_obs_fires} (should both be 0)")
    
    # Check ideal DEM
    try:
        # Inject minimal noise to generate DEM
        test_circ = ideal.copy()
        # Can't generate DEM from noiseless circuit; inject uniform depol
        noisy_test = stim.Circuit()
        for inst in ideal.flattened():
            noisy_test.append(inst)
            if inst.name in ("CX", "CZ", "CY", "ZCX", "ZCZ", "XCX"):
                noisy_test.append("DEPOLARIZE2", inst.targets_copy(), 0.001)
            elif inst.name in ("H", "S", "S_DAG", "H_YZ", "SQRT_X", "SQRT_X_DAG", "SQRT_Y", "SQRT_Y_DAG"):
                noisy_test.append("DEPOLARIZE1", inst.targets_copy(), 0.001)
        
        dem = noisy_test.detector_error_model(decompose_errors=True)
        n_dem_errors = sum(1 for inst in dem.flattened() if inst.type == "error")
        n_det_only = 0
        n_obs_only = 0
        n_both = 0
        n_empty = 0
        for inst in dem.flattened():
            if inst.type == "error":
                has_det = any(t.is_relative_detector_id() for t in inst.targets_copy())
                has_obs = any(t.is_logical_observable_id() for t in inst.targets_copy())
                if has_det and has_obs:
                    n_both += 1
                elif has_det:
                    n_det_only += 1
                elif has_obs:
                    n_obs_only += 1
                else:
                    n_empty += 1
        print(f"  DEM (uniform noise): {n_dem_errors} errors (det={n_det_only}, obs={n_obs_only}, both={n_both}, empty={n_empty})")
    except Exception as e:
        print(f"  DEM error: {type(e).__name__}: {e}")
    
    # Now get the actual hardware-noised circuit
    stim_circ = exp_t.to_stim()
    n_det = sum(1 for inst in stim_circ.flattened() if inst.name == "DETECTOR")
    n_obs = sum(1 for inst in stim_circ.flattened() if inst.name == "OBSERVABLE_INCLUDE")
    n_depol2 = sum(1 for inst in stim_circ.flattened() if inst.name == "DEPOLARIZE2")
    n_xerr = sum(1 for inst in stim_circ.flattened() if inst.name == "X_ERROR")
    n_zerr = sum(1 for inst in stim_circ.flattened() if inst.name == "Z_ERROR")
    print(f"  Hardware circuit: {stim_circ.num_qubits} qubits, {n_det} det, {n_obs} obs")
    print(f"  Noise: DEPOL2={n_depol2}, X_ERR={n_xerr}, Z_ERR={n_zerr}")
    
    # Check hardware DEM
    try:
        dem_hw = stim_circ.detector_error_model(decompose_errors=True)
        n_dem_hw = sum(1 for inst in dem_hw.flattened() if inst.type == "error")
        n_empty_hw = sum(1 for inst in dem_hw.flattened() 
                         if inst.type == "error" 
                         and not any(t.is_relative_detector_id() for t in inst.targets_copy())
                         and not any(t.is_logical_observable_id() for t in inst.targets_copy()))
        print(f"  DEM (hardware): {n_dem_hw} errors ({n_empty_hw} empty/undetectable)")
    except Exception as e:
        print(f"  DEM (hardware) error: {type(e).__name__}: {e}")


=== d=2 (r=2) ===
  Ideal circuit: 8 qubits, 4 detectors, 1 observables
  Noiseless ideal: det_fires=0, obs_fires=0 (should both be 0)
  DEM (uniform noise): 16 errors (det=9, obs=1, both=6, empty=0)


WISE routing: 100%|██████████| 10/10 [00:04<00:00,  2.07batch/s, pairs=2, ops=24]


  Hardware circuit: 8 qubits, 4 det, 1 obs
  Noise: DEPOL2=19, X_ERR=10, Z_ERR=0
  DEM (hardware): 16 errors (0 empty/undetectable)

=== d=3 (r=3) ===
  Ideal circuit: 18 qubits, 20 detectors, 1 observables
  Noiseless ideal: det_fires=0, obs_fires=0 (should both be 0)
  DEM (uniform noise): 108 errors (det=72, obs=1, both=35, empty=0)


WISE routing: 100%|██████████| 17/17 [00:30<00:00,  1.81s/batch, pairs=2, ops=150]


  Hardware circuit: 18 qubits, 20 det, 1 obs
  Noise: DEPOL2=32, X_ERR=20, Z_ERR=0
  DEM (hardware): 108 errors (0 empty/undetectable)

=== d=4 (r=4) ===
  Ideal circuit: 32 qubits, 52 detectors, 1 observables
  Noiseless ideal: det_fires=0, obs_fires=0 (should both be 0)
  DEM error: ValueError: Failed to decompose errors into graphlike components with at most two symptoms.
The error component that failed to decompose is 'D4, D16, D25, D26'.

In Python, you can ignore this error by passing `ignore_decomposition_failures=True` to `stim.Circuit.detector_error_model(...)`.
From the command line, you can ignore this error by passing the flag `--ignore_decomposition_failures` to `stim analyze_errors`.


WISE routing: 100%|██████████| 27/27 [02:38<00:00,  5.85s/batch, pairs=2, ops=419]

  Hardware circuit: 32 qubits, 52 det, 1 obs
  Noise: DEPOL2=47, X_ERR=35, Z_ERR=0
  DEM (hardware) error: ValueError: Failed to decompose errors into graphlike components with at most two symptoms.
The error component that failed to decompose is 'D4, D16, D25, D26'.

In Python, you can ignore this error by passing `ignore_decomposition_failures=True` to `stim.Circuit.detector_error_model(...)`.
From the command line, you can ignore this error by passing the flag `--ignore_decomposition_failures` to `stim analyze_errors`.


In [39]:
# ═══════════════════════════════════════════════════════════════════════
# T4 Deep Dive: Compare our d=4 circuit with Stim's built-in
# ═══════════════════════════════════════════════════════════════════════
import stim

# Stim's built-in rotated surface code circuit
stim_builtin = stim.Circuit.generated(
    "surface_code:rotated_memory_z", distance=4, rounds=4,
    after_clifford_depolarization=0.001,
    before_measure_flip_probability=0.001,
)

print("=== Stim built-in d=4 ===")
n_det_stim = sum(1 for inst in stim_builtin.flattened() if inst.name == "DETECTOR")
n_obs_stim = sum(1 for inst in stim_builtin.flattened() if inst.name == "OBSERVABLE_INCLUDE")
print(f"  Qubits: {stim_builtin.num_qubits}, Detectors: {n_det_stim}, Observables: {n_obs_stim}")
try:
    dem_stim = stim_builtin.detector_error_model(decompose_errors=True)
    print(f"  DEM: OK ({sum(1 for i in dem_stim.flattened() if i.type == 'error')} errors)")
except Exception as e:
    print(f"  DEM error: {e}")

# Compare gate types
for label, circ in [("Stim built-in", stim_builtin)]:
    gate_counts = {}
    for inst in circ.flattened():
        if inst.name not in ("TICK", "QUBIT_COORDS", "DETECTOR", "OBSERVABLE_INCLUDE", 
                              "SHIFT_COORDS", "REPEAT"):
            gate_counts[inst.name] = gate_counts.get(inst.name, 0) + 1
    print(f"\n  {label} gates: {dict(sorted(gate_counts.items()))}")

# Now our d=4 circuit
code_4 = RotatedSurfaceCode(4)
from qectostim.experiments.memory import CSSMemoryExperiment
css_exp = CSSMemoryExperiment(code=code_4, noise_model=None, rounds=4, basis="Z")
our_circ = css_exp.to_stim()

print(f"\n=== Our d=4 (CSSMemoryExperiment) ===")
n_det_ours = sum(1 for inst in our_circ.flattened() if inst.name == "DETECTOR")
n_obs_ours = sum(1 for inst in our_circ.flattened() if inst.name == "OBSERVABLE_INCLUDE")
print(f"  Qubits: {our_circ.num_qubits}, Detectors: {n_det_ours}, Observables: {n_obs_ours}")

gate_counts_ours = {}
for inst in our_circ.flattened():
    if inst.name not in ("TICK", "QUBIT_COORDS", "DETECTOR", "OBSERVABLE_INCLUDE",
                          "SHIFT_COORDS", "REPEAT"):
        gate_counts_ours[inst.name] = gate_counts_ours.get(inst.name, 0) + 1
print(f"  Our gates: {dict(sorted(gate_counts_ours.items()))}")

# Inject uniform noise to test DEM
our_noisy = stim.Circuit()
for inst in our_circ.flattened():
    our_noisy.append(inst)
    if inst.name in ("CX", "CZ", "CY", "ZCX", "ZCZ", "XCX", "CNOT"):
        our_noisy.append("DEPOLARIZE2", inst.targets_copy(), 0.001)
    elif inst.name in ("H", "S", "S_DAG", "H_YZ", "SQRT_X", "SQRT_X_DAG"):
        our_noisy.append("DEPOLARIZE1", inst.targets_copy(), 0.001)

try:
    dem_ours = our_noisy.detector_error_model(decompose_errors=True)
    print(f"  DEM: OK ({sum(1 for i in dem_ours.flattened() if i.type == 'error')} errors)")
except ValueError as e:
    print(f"  DEM error: {str(e)[:200]}")
    
    # Try with ignore_decomposition_failures
    dem_ours_lax = our_noisy.detector_error_model(
        decompose_errors=True, ignore_decomposition_failures=True
    )
    n_errs = sum(1 for i in dem_ours_lax.flattened() if i.type == "error")
    # Count hyperedge errors (>2 symptoms)
    n_hyper = 0
    for inst in dem_ours_lax.flattened():
        if inst.type == "error":
            n_dets = sum(1 for t in inst.targets_copy() if t.is_relative_detector_id())
            if n_dets > 2:
                n_hyper += 1
    print(f"  DEM (lenient): {n_errs} errors, {n_hyper} hyperedge errors")

# Check if the issue is CZ vs CX
# Force CX mode
css_exp_cx = CSSMemoryExperiment(code=code_4, noise_model=None, rounds=4, basis="Z")
# Need to override x_stabilizer_mode - check if CSSMemoryExperiment passes it through
print(f"\n  Our x_stabilizer_mode: 'cz' (default for Z-basis memory)")

=== Stim built-in d=4 ===
  Qubits: 43, Detectors: 59, Observables: 1
  DEM: OK (872 errors)

  Stim built-in gates: {'CX': 16, 'DEPOLARIZE1': 8, 'DEPOLARIZE2': 16, 'H': 8, 'M': 1, 'MR': 4, 'R': 1, 'X_ERROR': 5}

=== Our d=4 (CSSMemoryExperiment) ===
  Qubits: 32, Detectors: 52, Observables: 1
  Our gates: {'CX': 31, 'CZ': 16, 'H': 24, 'M': 16, 'MR': 4, 'R': 16}
  DEM error: Failed to decompose errors into graphlike components with at most two symptoms.
The error component that failed to decompose is 'D4, D16, D25, D26'.

In Python, you can ignore this error by passing `i
  DEM (lenient): 267 errors, 125 hyperedge errors

  Our x_stabilizer_mode: 'cz' (default for Z-basis memory)


In [40]:
# ═══════════════════════════════════════════════════════════════════════
# T4 Root Cause: CZ vs CX for X-stabilizer measurement
# ═══════════════════════════════════════════════════════════════════════
# Stim uses CX (CNOT) for all stabilizers. Our code uses CZ for X stabilizers.
# CZ in an interleaved schedule creates non-graphlike hook errors.
# Test: manually force "cx" mode by patching the builder

from qectostim.experiments.memory import CSSMemoryExperiment
from qectostim.experiments.stabilizer_rounds.css import CSSStabilizerRoundBuilder
from qectostim.experiments.stabilizer_rounds.base import DetectorContext, StabilizerBasis

# Manually build circuit with x_stabilizer_mode="cx"
code_4 = RotatedSurfaceCode(4)
ctx = DetectorContext()
builder = CSSStabilizerRoundBuilder(
    code_4, ctx, block_name="main", measurement_basis="Z",
    x_stabilizer_mode="cx",  # Force CX instead of CZ
)

c = stim.Circuit()
builder.emit_qubit_coords(c)
builder.emit_reset_all(c)
builder.emit_prepare_logical_state(c, state="0", logical_idx=0)

# First round
builder.emit_round(c, stab_type=StabilizerBasis.BOTH, emit_detectors=True)

# REPEAT block for rounds 2-4
repeat_body = stim.Circuit()
builder.emit_round(repeat_body, stab_type=StabilizerBasis.BOTH, emit_detectors=True)
c.append(stim.CircuitRepeatBlock(3, repeat_body))

# Final measurement
builder.emit_final_measurement(c, basis="Z", logical_idx=0)
ctx.emit_observable(c, observable_idx=0)

# Check
n_det_cx = sum(1 for inst in c.flattened() if inst.name == "DETECTOR")
n_obs_cx = sum(1 for inst in c.flattened() if inst.name == "OBSERVABLE_INCLUDE")
print(f"=== Our d=4 with CX mode ===")
print(f"  Qubits: {c.num_qubits}, Detectors: {n_det_cx}, Observables: {n_obs_cx}")

# Count actual gates
gate_counts = {}
for inst in c.flattened():
    if inst.name in ("CX", "CZ", "CNOT", "H", "MR", "M", "R"):
        # Count individual gates (each target pair = 1 gate)
        if inst.name in ("CX", "CZ", "CNOT"):
            gate_counts[inst.name] = gate_counts.get(inst.name, 0) + len(inst.targets_copy()) // 2
        else:
            gate_counts[inst.name] = gate_counts.get(inst.name, 0) + len(inst.targets_copy())
print(f"  Gate counts: {dict(sorted(gate_counts.items()))}")

# Inject uniform noise
noisy_cx = stim.Circuit()
for inst in c.flattened():
    noisy_cx.append(inst)
    if inst.name in ("CX", "CNOT"):
        for i in range(0, len(inst.targets_copy()), 2):
            targets = inst.targets_copy()
            noisy_cx.append("DEPOLARIZE2", [targets[i], targets[i+1]], 0.001)
    elif inst.name == "H":
        noisy_cx.append("DEPOLARIZE1", inst.targets_copy(), 0.001)

try:
    dem_cx = noisy_cx.detector_error_model(decompose_errors=True)
    n_errs = sum(1 for i in dem_cx.flattened() if i.type == "error")
    print(f"  DEM: OK! ({n_errs} errors) ← NO hyperedge errors!")
except ValueError as e:
    print(f"  DEM error: {str(e)[:200]}")

# Noiseless decode test
det_sampler = c.compile_detector_sampler()
det_samples = det_sampler.sample(100, append_observables=True)
n_det_fires = det_samples[:, :n_det_cx].sum()
n_obs_fires = det_samples[:, n_det_cx:].sum()
print(f"  Noiseless: det_fires={n_det_fires}, obs_fires={n_obs_fires} (should both be 0)")

# Test decoding with uniform noise
noisy_cx_test = stim.Circuit()
for inst in c.flattened():
    noisy_cx_test.append(inst)
    if inst.name in ("CX", "CNOT"):
        for i in range(0, len(inst.targets_copy()), 2):
            targets = inst.targets_copy()
            noisy_cx_test.append("DEPOLARIZE2", [targets[i], targets[i+1]], 0.01)  # 1% error
    elif inst.name == "H":
        noisy_cx_test.append("DEPOLARIZE1", inst.targets_copy(), 0.01)

import pymatching
dem_test = noisy_cx_test.detector_error_model(decompose_errors=True)
matcher = pymatching.Matching.from_detector_error_model(dem_test)
sampler = noisy_cx_test.compile_detector_sampler()
samples = sampler.sample(2000, append_observables=True)
predictions = matcher.decode_batch(samples[:, :-1])
n_errors = (predictions != samples[:, -1:]).sum()
ler = n_errors / 2000
print(f"  d=4 CX-mode LER at 1% noise: {ler:.4f}")

# Compare with d=2 at same noise
code_2 = RotatedSurfaceCode(2)
ctx2 = DetectorContext()
builder2 = CSSStabilizerRoundBuilder(code_2, ctx2, block_name="main", measurement_basis="Z", x_stabilizer_mode="cx")
c2 = stim.Circuit()
builder2.emit_qubit_coords(c2)
builder2.emit_reset_all(c2)
builder2.emit_prepare_logical_state(c2, state="0", logical_idx=0)
builder2.emit_round(c2, stab_type=StabilizerBasis.BOTH, emit_detectors=True)
repeat2 = stim.Circuit()
builder2.emit_round(repeat2, stab_type=StabilizerBasis.BOTH, emit_detectors=True)
c2.append(stim.CircuitRepeatBlock(1, repeat2))
builder2.emit_final_measurement(c2, basis="Z", logical_idx=0)
ctx2.emit_observable(c2, observable_idx=0)

noisy_c2 = stim.Circuit()
for inst in c2.flattened():
    noisy_c2.append(inst)
    if inst.name in ("CX", "CNOT"):
        for i in range(0, len(inst.targets_copy()), 2):
            targets = inst.targets_copy()
            noisy_c2.append("DEPOLARIZE2", [targets[i], targets[i+1]], 0.01)
    elif inst.name == "H":
        noisy_c2.append("DEPOLARIZE1", inst.targets_copy(), 0.01)

dem2 = noisy_c2.detector_error_model(decompose_errors=True)
matcher2 = pymatching.Matching.from_detector_error_model(dem2)
sampler2 = noisy_c2.compile_detector_sampler()
samples2 = sampler2.sample(2000, append_observables=True)
predictions2 = matcher2.decode_batch(samples2[:, :-1])
n_errors2 = (predictions2 != samples2[:, -1:]).sum()
ler2 = n_errors2 / 2000
print(f"  d=2 CX-mode LER at 1% noise: {ler2:.4f}")
print(f"  → d=4 < d=2? {ler < ler2} (this is the T4 trend we need!)")

=== Our d=4 with CX mode ===
  Qubits: 32, Detectors: 52, Observables: 1
  Gate counts: {'CX': 240, 'H': 80, 'M': 31, 'MR': 60, 'R': 46}
  DEM: OK! (833 errors) ← NO hyperedge errors!
  Noiseless: det_fires=0, obs_fires=0 (should both be 0)
  d=4 CX-mode LER at 1% noise: 0.1590
  d=2 CX-mode LER at 1% noise: 0.0875
  → d=4 < d=2? False (this is the T4 trend we need!)


In [41]:
# ═══════════════════════════════════════════════════════════════════════
# T4 Confirmation: CX mode below threshold → d=4 < d=2 
# ═══════════════════════════════════════════════════════════════════════
import pymatching

for p_noise in [0.005, 0.003, 0.001]:
    results = {}
    for d_val in [2, 4]:
        code_t = RotatedSurfaceCode(d_val)
        ctx_t = DetectorContext()
        builder_t = CSSStabilizerRoundBuilder(
            code_t, ctx_t, block_name="main", measurement_basis="Z", 
            x_stabilizer_mode="cx"
        )
        c_t = stim.Circuit()
        builder_t.emit_qubit_coords(c_t)
        builder_t.emit_reset_all(c_t)
        builder_t.emit_prepare_logical_state(c_t, state="0", logical_idx=0)
        builder_t.emit_round(c_t, stab_type=StabilizerBasis.BOTH, emit_detectors=True)
        if d_val > 2:
            rb = stim.Circuit()
            builder_t.emit_round(rb, stab_type=StabilizerBasis.BOTH, emit_detectors=True)
            c_t.append(stim.CircuitRepeatBlock(d_val - 1, rb))
        builder_t.emit_final_measurement(c_t, basis="Z", logical_idx=0)
        ctx_t.emit_observable(c_t, observable_idx=0)
        
        noisy_t = stim.Circuit()
        for inst in c_t.flattened():
            noisy_t.append(inst)
            if inst.name in ("CX", "CNOT"):
                for i in range(0, len(inst.targets_copy()), 2):
                    targets = inst.targets_copy()
                    noisy_t.append("DEPOLARIZE2", [targets[i], targets[i+1]], p_noise)
            elif inst.name == "H":
                noisy_t.append("DEPOLARIZE1", inst.targets_copy(), p_noise)
        
        dem_t = noisy_t.detector_error_model(decompose_errors=True)
        matcher_t = pymatching.Matching.from_detector_error_model(dem_t)
        sampler_t = noisy_t.compile_detector_sampler()
        samples_t = sampler_t.sample(10000, append_observables=True)
        predictions_t = matcher_t.decode_batch(samples_t[:, :-1])
        n_err = (predictions_t != samples_t[:, -1:]).sum()
        results[d_val] = n_err / 10000
    
    trend = "✅" if results[4] < results[2] else "❌"
    print(f"p={p_noise:.3f}: d=2 LER={results[2]:.4f}, d=4 LER={results[4]:.4f} {trend}")

p=0.005: d=2 LER=0.0222, d=4 LER=0.0678 ❌
p=0.003: d=2 LER=0.0125, d=4 LER=0.0365 ❌
p=0.001: d=2 LER=0.0038, d=4 LER=0.0126 ❌


In [42]:
# ═══════════════════════════════════════════════════════════════════════
# T4: Compare with Stim built-in at same noise levels
# ═══════════════════════════════════════════════════════════════════════

for p_noise in [0.005, 0.003, 0.001]:
    results = {}
    for d_val in [2, 4]:
        circ = stim.Circuit.generated(
            "surface_code:rotated_memory_z", distance=d_val, rounds=d_val,
            after_clifford_depolarization=p_noise,
            before_measure_flip_probability=p_noise,
        )
        dem = circ.detector_error_model(decompose_errors=True)
        matcher = pymatching.Matching.from_detector_error_model(dem)
        sampler = circ.compile_detector_sampler()
        samples = sampler.sample(10000, append_observables=True)
        preds = matcher.decode_batch(samples[:, :-1])
        ler = (preds != samples[:, -1:]).sum() / 10000
        results[d_val] = ler
    trend = "✅" if results[4] < results[2] else "❌"
    print(f"STIM p={p_noise:.3f}: d=2 LER={results[2]:.4f}, d=4 LER={results[4]:.4f} {trend}")

STIM p=0.005: d=2 LER=0.0277, d=4 LER=0.0084 ✅
STIM p=0.003: d=2 LER=0.0188, d=4 LER=0.0037 ✅
STIM p=0.001: d=2 LER=0.0069, d=4 LER=0.0004 ✅


In [43]:
# ═══════════════════════════════════════════════════════════════════════
# Compare circuit structure: Stim built-in vs Our CSSMemoryExperiment
# ═══════════════════════════════════════════════════════════════════════
# Print first round of each d=4 circuit

# Stim's built-in (noiseless)
stim_noiseless = stim.Circuit.generated(
    "surface_code:rotated_memory_z", distance=4, rounds=4,
    after_clifford_depolarization=0.0,
    before_measure_flip_probability=0.0,
)
print("=== STIM BUILT-IN d=4 (first 60 instructions) ===")
lines = str(stim_noiseless).split('\n')
for i, line in enumerate(lines[:60]):
    print(f"  {i:3d}: {line}")

print(f"\n  ... ({len(lines)} total lines)")
print(f"  Qubits: {stim_noiseless.num_qubits}")
n_det_stim = sum(1 for inst in stim_noiseless.flattened() if inst.name == "DETECTOR")
print(f"  Detectors: {n_det_stim}")

print("\n" + "="*70)

# Our circuit with CX mode
print("\n=== OUR d=4 CX MODE (first 60 instructions) ===")
our_lines = str(c).split('\n')
for i, line in enumerate(our_lines[:60]):
    print(f"  {i:3d}: {line}")
print(f"\n  ... ({len(our_lines)} total lines)")
print(f"  Qubits: {c.num_qubits}")

=== STIM BUILT-IN d=4 (first 60 instructions) ===
    0: QUBIT_COORDS(1, 1) 1
    1: QUBIT_COORDS(2, 0) 2
    2: QUBIT_COORDS(3, 1) 3
    3: QUBIT_COORDS(5, 1) 5
    4: QUBIT_COORDS(6, 0) 6
    5: QUBIT_COORDS(7, 1) 7
    6: QUBIT_COORDS(1, 3) 10
    7: QUBIT_COORDS(2, 2) 11
    8: QUBIT_COORDS(3, 3) 12
    9: QUBIT_COORDS(4, 2) 13
   10: QUBIT_COORDS(5, 3) 14
   11: QUBIT_COORDS(6, 2) 15
   12: QUBIT_COORDS(7, 3) 16
   13: QUBIT_COORDS(0, 4) 18
   14: QUBIT_COORDS(1, 5) 19
   15: QUBIT_COORDS(2, 4) 20
   16: QUBIT_COORDS(3, 5) 21
   17: QUBIT_COORDS(4, 4) 22
   18: QUBIT_COORDS(5, 5) 23
   19: QUBIT_COORDS(6, 4) 24
   20: QUBIT_COORDS(7, 5) 25
   21: QUBIT_COORDS(8, 4) 26
   22: QUBIT_COORDS(1, 7) 28
   23: QUBIT_COORDS(2, 6) 29
   24: QUBIT_COORDS(3, 7) 30
   25: QUBIT_COORDS(4, 6) 31
   26: QUBIT_COORDS(5, 7) 32
   27: QUBIT_COORDS(6, 6) 33
   28: QUBIT_COORDS(7, 7) 34
   29: QUBIT_COORDS(2, 8) 38
   30: QUBIT_COORDS(6, 8) 42
   31: R 1 3 5 7 10 12 14 16 19 21 23 25 28 30 32 34 2 6 

In [ ]:
# Print full OUR circuit structure
print("=== OUR d=4 CX MODE (FULL) ===")
our_lines = str(c).split('\n')
for i, line in enumerate(our_lines):
    print(f"  {i:3d}: {line}")

In [45]:
# ── Force deep reload after code changes ──────────────────────────────────────
import importlib, sys

# Remove all cached qectostim modules so changes are picked up
mods_to_remove = [k for k in sys.modules if k.startswith("qectostim")]
for m in mods_to_remove:
    del sys.modules[m]

# Re-import everything fresh
import time, numpy as np, pandas as pd
from qectostim.codes.surface.rotated_surface import RotatedSurfaceCode
from qectostim.experiments.memory import CSSMemoryExperiment
from qectostim.experiments.hardware_simulation.trapped_ion.architecture import WISEArchitecture
from qectostim.experiments.hardware_simulation.trapped_ion.compiler import WISECompiler
from qectostim.experiments.hardware_simulation.trapped_ion.experiments import TrappedIonExperiment
from qectostim.experiments.hardware_simulation.trapped_ion.noise import TrappedIonNoiseModel
from qectostim.experiments.hardware_simulation.trapped_ion.routing import WISERoutingConfig

print("✅ Modules reloaded (ElapsedTime + multi-round advancement fix)")

# ── 8 representative configs (same as before) ────────────────────────────────
REPRESENTATIVE = [
    # (d, k, la, sg_area, sg_inc, label)
    (2, 2, 1, "full", 1, "d2_k2_la1_full"),   # Baseline
    (2, 2, 4, "full", 1, "d2_k2_la4_full"),   # T1: la↑ → exec↓
    (2, 2, 1, 6,      1, "d2_k2_la1_small"),  # T2: small grid → exec↑
    (2, 4, 1, "full", 1, "d2_k4_la1_full"),   # T3a: k↑ → ops same
    (2, 8, 1, "full", 1, "d2_k8_la1_full"),   # T3b: k=8 → ops↑
    (3, 2, 1, "full", 1, "d3_k2_la1_full"),   # T5/T6 baseline
    (3, 2, 2, "full", 1, "d3_k2_la2_full"),   # T5: la↑ → exec↓
    (4, 2, 1, "full", 1, "d4_k2_la1_full"),   # T6: d↑ → exec↑
]

OLD_RESULTS = [
    {"label": "d2_k2_la1_full",  "exec": 5659,  "ler": 0.1161},
    {"label": "d2_k2_la4_full",  "exec": 3674,  "ler": 0.1153},
    {"label": "d2_k2_la1_small", "exec": 9787,  "ler": 0.1189},
    {"label": "d2_k4_la1_full",  "exec": 4840,  "ler": 0.1169},
    {"label": "d2_k8_la1_full",  "exec": 5032,  "ler": 0.1481},
    {"label": "d3_k2_la1_full",  "exec": 7649,  "ler": 0.1235},
    {"label": "d3_k2_la2_full",  "exec": 5983,  "ler": 0.1211},
    {"label": "d4_k2_la1_full",  "exec": 9052,  "ler": 0.0920},
]

NUM_VAL_SHOTS = 1000
SAT_TIMEOUT = 30.0

new_results = []
for d, k, la, sg, sg_inc, label in REPRESENTATIVE:
    t0 = time.time()
    code = RotatedSurfaceCode(d)
    n_q = code.n
    n_traps = int(np.ceil(np.sqrt(4 * np.ceil(n_q / 3))))
    m = int(np.ceil(n_traps / k))
    n = n_traps

    arch = WISEArchitecture(col_groups=m, rows=n, ions_per_segment=k)

    if sg == "full":
        sw, sh = m * k, n
    else:
        sw = sh = sg

    routing_cfg = WISERoutingConfig(
        subgridsize=(sw, sh, sg_inc),
        lookahead_rounds=la,
        timeout_seconds=SAT_TIMEOUT,
    )
    compiler = WISECompiler(architecture=arch, routing_config=routing_cfg)
    noise = TrappedIonNoiseModel()

    exp = TrappedIonExperiment(
        code=code,
        architecture=arch,
        compiler=compiler,
        noise_model=noise,
        rounds=d,
    )
    result = exp.simulate(num_shots=NUM_VAL_SHOTS)
    wall = time.time() - t0

    sm = result.simulation_metrics
    new_exec = sm.get("ElapsedTime", 0.0)
    new_reconfig = sm.get("ReconfigurationTime", 0.0)
    new_ops = sm.get("Operations", 0)
    new_2q = sm.get("QubitOperations", 0)
    num_batches = result.compilation_metrics.get("num_batches", "?")

    new_results.append({
        "label": label,
        "exec": new_exec,
        "reconfig": new_reconfig,
        "ler": result.logical_error_rate,
        "ops": new_ops,
        "2q": new_2q,
        "batches": num_batches,
    })
    print(f"  {label:22s}  exec={new_exec:8.0f}µs  reconfig={new_reconfig:8.0f}µs  "
          f"LER={result.logical_error_rate:.4f}  ops={new_ops:4d}  "
          f"2q={new_2q:3d}  batches={num_batches}  [{wall:.1f}s]")

# ── Compare with old results ─────────────────────────────────────────────────
print("\n" + "="*90)
print(f"{'Config':<22s}  {'Old exec':>10s}  {'New exec':>10s}  {'Ratio':>8s}  "
      f"{'Old LER':>10s}  {'New LER':>10s}")
print("-"*90)
for old, new in zip(OLD_RESULTS, new_results):
    ratio = new["exec"] / old["exec"] if old["exec"] else float("inf")
    print(f"  {old['label']:<22s}  {old['exec']:>8.0f}µs  {new['exec']:>8.0f}µs  "
          f"{ratio:>7.2f}x  {old['ler']:>9.4f}  {new['ler']:>9.4f}")

# ── Check trends ─────────────────────────────────────────────────────────────
print("\n" + "="*90)
trends = [
    ("T1: la↑→exec↓ (d=2)",
     new_results[1]["exec"] < new_results[0]["exec"],
     f"{new_results[0]['exec']:.0f} → {new_results[1]['exec']:.0f}"),
    ("T2: small→exec↑ (d=2)",
     new_results[2]["exec"] > new_results[0]["exec"],
     f"full={new_results[0]['exec']:.0f} small={new_results[2]['exec']:.0f}"),
    ("T3a: k↑→ops≈ (k=4)",
     abs(new_results[3]["ops"] - new_results[0]["ops"]) < 50,
     f"k2={new_results[0]['ops']} k4={new_results[3]['ops']}"),
    ("T3b: k=8→ops↑",
     new_results[4]["ops"] > new_results[0]["ops"],
     f"k2={new_results[0]['ops']} k8={new_results[4]['ops']}"),
    ("T5: la↑→exec↓ (d=3)",
     new_results[6]["exec"] < new_results[5]["exec"],
     f"{new_results[5]['exec']:.0f} → {new_results[6]['exec']:.0f}"),
    ("T6: d↑→exec↑",
     new_results[5]["exec"] > new_results[0]["exec"],
     f"d2={new_results[0]['exec']:.0f} d3={new_results[5]['exec']:.0f}"),
    ("T4: d↑→LER↓",
     new_results[7]["ler"] < new_results[0]["ler"],
     f"d2={new_results[0]['ler']:.4f} d4={new_results[7]['ler']:.4f}"),
]
pass_count = 0
for name, passed, detail in trends:
    status = "✅" if passed else "❌"
    if passed:
        pass_count += 1
    print(f"  {status} {name:<30s}  {detail}")
print(f"\nScore: {pass_count}/{len(trends)} trends passing")

✅ Modules reloaded (ElapsedTime + multi-round advancement fix)
  [1/4] Building ideal circuit …
         → 8Q, 46 instructions
  [3/4] Compiling (decompose → map → route → schedule) …


WISE routing: 100%|██████████| 10/10 [00:01<00:00,  5.71batch/s, pairs=1, ops=34]

         → compiled OK
  [4/4] Decoding (1000 shots) …
  ✅ Done in 1.82s — LER=3.9000e-02
  d2_k2_la1_full          exec=    7804µs  reconfig=    6668µs  LER=0.0390  ops= 229  2q= 19  batches=10  [1.8s]
  [1/4] Building ideal circuit …
         → 8Q, 46 instructions
  [3/4] Compiling (decompose → map → route → schedule) …



WISE routing: 100%|██████████| 10/10 [00:09<00:00,  1.06batch/s, pairs=3, ops=39]

         → compiled OK
  [4/4] Decoding (1000 shots) …
  ✅ Done in 9.53s — LER=4.7000e-02
  d2_k2_la4_full          exec=   13588µs  reconfig=   12352µs  LER=0.0470  ops= 325  2q= 19  batches=10  [9.5s]
  [1/4] Building ideal circuit …
         → 8Q, 46 instructions
  [3/4] Compiling (decompose → map → route → schedule) …


...

WISE routing: 100%|██████████| 17/17 [00:16<00:00,  1.06batch/s, pairs=1, ops=158]

         → compiled OK
  [4/4] Decoding (1000 shots) …
  ✅ Done in 16.15s — LER=5.8000e-02
  d3_k2_la2_full          exec=   23272µs  reconfig=   21536µs  LER=0.0580  ops= 601  2q= 32  batches=17  [16.2s]
  [1/4] Building ideal circuit …


         → 32Q, 124 instructions
  [3/4] Compiling (decompose → map → route → schedule) …


WISE routing: 100%|██████████| 27/27 [00:29<00:00,  1.09s/batch, pairs=2, ops=364]

         → compiled OK
  [4/4] Decoding (1000 shots) …
  ✅ Done in 29.63s — LER=8.3000e-02
  d4_k2_la1_full          exec=   28596µs  reconfig=   26066µs  LER=0.0830  ops=1170  2q= 47  batches=27  [29.6s]

Config                    Old exec    New exec     Ratio     Old LER     New LER
------------------------------------------------------------------------------------------
  d2_k2_la1_full              5659µs      7804µs     1.38x     0.1161     0.0390
  d2_k2_la4_full              3674µs     13588µs     3.70x     0.1153     0.0470
  d2_k2_la1_small             9787µs      7804µs     0.80x     0.1189     0.0440
  d2_k4_la1_full              4840µs      7804µs     1.61x     0.1169     0.0470
  d2_k8_la1_full              5032µs      6500µs     1.29x     0.1481     0.0550
  d3_k2_la1_full              7649µs     21328µs     2.79x     0.1235     0.0700
  d3_k2_la2_full              5983µs     23272µs     3.89x     0.1211     0.0580
  d4_k2_la1_full              9052µs     28596µs     3.

In [47]:
# Diagnostic: trace route_batch calls to verify multi-round decoding
import importlib, sys
mods_to_remove = [k for k in sys.modules if k.startswith("qectostim")]
for m in mods_to_remove:
    del sys.modules[m]

import time, numpy as np
from qectostim.codes.surface.rotated_surface import RotatedSurfaceCode
from qectostim.experiments.hardware_simulation.trapped_ion.architecture import WISEArchitecture
from qectostim.experiments.hardware_simulation.trapped_ion.compiler import WISECompiler
from qectostim.experiments.hardware_simulation.trapped_ion.experiments import TrappedIonExperiment
from qectostim.experiments.hardware_simulation.trapped_ion.noise import TrappedIonNoiseModel
from qectostim.experiments.hardware_simulation.trapped_ion.routing import (
    WISERoutingConfig, WiseSatRouter, WISERoutingPass,
)

# Monkey-patch route_batch to trace calls
_orig_route_batch = WiseSatRouter.route_batch
_call_log = []

def _traced_route_batch(self, gate_pairs, current_mapping, architecture, lookahead_pairs=None):
    result = _orig_route_batch(self, gate_pairs, current_mapping, architecture, lookahead_pairs=lookahead_pairs)
    metrics = result.metrics or {}
    num_decoded = metrics.get("num_rounds_decoded", "?")
    schedule = metrics.get("_schedule")
    ppr = len(schedule.passes_per_round) if schedule else 0
    entry = {
        "pairs": len(gate_pairs),
        "lookahead": None if lookahead_pairs is None else len(lookahead_pairs),
        "num_decoded": num_decoded,
        "schedule_rounds": ppr,
        "success": result.success,
    }
    _call_log.append(entry)
    return result

WiseSatRouter.route_batch = _traced_route_batch

# d=2, k=2, la=4 — the case where we expect multi-round advancement
code = RotatedSurfaceCode(2)
n_q = code.n
n_traps = int(np.ceil(np.sqrt(4 * np.ceil(n_q / 3))))
k = 2
m_val = int(np.ceil(n_traps / k))
arch = WISEArchitecture(col_groups=m_val, rows=n_traps, ions_per_segment=k)

routing_cfg = WISERoutingConfig(
    subgridsize=(m_val * k, n_traps, 1),
    lookahead_rounds=4,
    timeout_seconds=30.0,
)
compiler = WISECompiler(architecture=arch, routing_config=routing_cfg)
noise = TrappedIonNoiseModel()

exp = TrappedIonExperiment(code=code, architecture=arch, compiler=compiler, noise_model=noise, rounds=2)
_call_log.clear()
result = exp.simulate(num_shots=100)

sm = result.simulation_metrics
print(f"ElapsedTime={sm.get('ElapsedTime', 0):.0f}µs  "
      f"ReconfigTime={sm.get('ReconfigurationTime', 0):.0f}µs")
print(f"\nroute_batch calls: {len(_call_log)}")
for i, entry in enumerate(_call_log):
    print(f"  call {i}: pairs={entry['pairs']}, "
          f"lookahead={entry['lookahead']}, "
          f"num_decoded={entry['num_decoded']}, "
          f"schedule_rounds={entry['schedule_rounds']}, "
          f"success={entry['success']}")

# Restore
WiseSatRouter.route_batch = _orig_route_batch

  [1/4] Building ideal circuit …
         → 8Q, 46 instructions
  [3/4] Compiling (decompose → map → route → schedule) …


WISE routing: 100%|██████████| 10/10 [00:08<00:00,  1.21batch/s, pairs=3, ops=45]

         → compiled OK
  [4/4] Decoding (100 shots) …
  ✅ Done in 8.32s — LER=6.0000e-02
ElapsedTime=13588µs  ReconfigTime=12352µs

route_batch calls: 2
  call 0: pairs=1, lookahead=4, num_decoded=5, schedule_rounds=5, success=True
  call 1: pairs=3, lookahead=4, num_decoded=5, schedule_rounds=5, success=True


In [35]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 5: Comparison table — Old vs New
# ═══════════════════════════════════════════════════════════════════════
df_ok = df_new[df_new["status"] == "ok"].copy()

if len(df_ok) == 0:
    print("⚠️  No configs succeeded — cannot build comparison")
else:
    df_ok["exec_ratio"] = df_ok["new_exec_us"] / df_ok["old_exec_us"]
    df_ok["ler_ratio"]  = df_ok["new_ler"] / df_ok["old_ler"]

    print("=" * 120)
    print(f"{'Label':<22} │ {'Old exec µs':>12} {'New exec µs':>12} {'Ratio':>7} │"
          f" {'Old LER':>9} {'New LER':>9} {'Ratio':>7} │ {'Ops':>5} {'2Q':>4} │ {'Wall':>6}")
    print("─" * 120)
    for _, row in df_ok.iterrows():
        print(f"{row['label']:<22} │ {row['old_exec_us']:>12.0f} {row['new_exec_us']:>12.0f} {row['exec_ratio']:>7.2f} │"
              f" {row['old_ler']:>9.4f} {row['new_ler']:>9.4f} {row['ler_ratio']:>7.2f} │"
              f" {row.get('new_ops', '?'):>5} {row.get('new_2q_ops', '?'):>4} │ {row['wall_s']:>5.1f}s")
    print("─" * 120)
    print(f"  exec_ratio: new/old (values near 1.0 = close match)")
    print(f"  ler_ratio:  new/old (absolute may differ; trend direction matters)")
    print(f"\n  Mean exec ratio: {df_ok['exec_ratio'].mean():.2f} ± {df_ok['exec_ratio'].std():.2f}")
    print(f"  Mean LER ratio:  {df_ok['ler_ratio'].mean():.2f} ± {df_ok['ler_ratio'].std():.2f}")

Label                  │  Old exec µs  New exec µs   Ratio │   Old LER   New LER   Ratio │   Ops   2Q │   Wall
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
d2_k2_la1_full         │         5659         1086    0.19 │    0.1161    0.0825    0.71 │   208   19 │   4.8s
d2_k2_la4_full         │         3674         1086    0.30 │    0.1153    0.0965    0.84 │   220   19 │  37.7s
d2_k2_la1_small        │         9787         1086    0.11 │    0.1189    0.0825    0.69 │   208   19 │   4.7s
d2_k4_la1_full         │         4840         1086    0.22 │    0.1169    0.1015    0.87 │   208   19 │   4.7s
d2_k8_la1_full         │         5032         1166    0.23 │    0.1481    0.2160    1.46 │   364   19 │  18.9s
d3_k2_la1_full         │         7649         1656    0.22 │    0.1235    0.1610    1.30 │   700   32 │  48.7s
d3_k2_la2_full         │         5983         1626    0.27 │    0.1211    0.1415    1.17 │   582   32 

In [36]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 6: Trend verification — directional trend checks
# ═══════════════════════════════════════════════════════════════════════

def get_val(label: str, col: str):
    rows = df_ok[df_ok["label"] == label]
    return rows.iloc[0][col] if len(rows) > 0 else None

trend_results = []

def check_trend(name, desc, label_a, label_b, col_old, col_new, expect_a_lt_b):
    old_a = get_val(label_a, col_old)
    old_b = get_val(label_b, col_old)
    new_a = get_val(label_a, col_new)
    new_b = get_val(label_b, col_new)
    if any(v is None for v in [old_a, old_b, new_a, new_b]):
        trend_results.append((name, desc, "SKIP", "missing data"))
        return
    if expect_a_lt_b:
        old_ok = old_a < old_b
        new_ok = new_a < new_b
    else:
        old_ok = old_a > old_b
        new_ok = new_a > new_b
    op = "<" if expect_a_lt_b else ">"
    match = "✅ MATCH" if old_ok == new_ok else "❌ MISMATCH"
    detail = (f"old: {old_a:.2f} {op} {old_b:.2f} = {old_ok}  |  "
              f"new: {new_a:.2f} {op} {new_b:.2f} = {new_ok}")
    trend_results.append((name, desc, match, detail))

# T1: Lookahead ↑ → exec_time ↓
check_trend("T1", "la↑ → exec↓ (d=2,k=2)",
            "d2_k2_la4_full", "d2_k2_la1_full",
            "old_exec_us", "new_exec_us", expect_a_lt_b=True)

# T2: Full-grid → lower exec_time than small subgrid
check_trend("T2", "full-grid → exec↓ (d=2,k=2,la=1)",
            "d2_k2_la1_full", "d2_k2_la1_small",
            "old_exec_us", "new_exec_us", expect_a_lt_b=True)

# T3a: k↑ → LER↑ (k=2 vs k=4)
check_trend("T3a", "k↑ → LER↑ (k2 vs k4)",
            "d2_k2_la1_full", "d2_k4_la1_full",
            "old_ler", "new_ler", expect_a_lt_b=True)

# T3b: k↑ → LER↑ (k=4 vs k=8)
check_trend("T3b", "k↑ → LER↑ (k4 vs k8)",
            "d2_k4_la1_full", "d2_k8_la1_full",
            "old_ler", "new_ler", expect_a_lt_b=True)

# T4: d↑ → LER↓ (d=4 < d=2)
check_trend("T4", "d↑ → LER↓ (d4 vs d2)",
            "d4_k2_la1_full", "d2_k2_la1_full",
            "old_ler", "new_ler", expect_a_lt_b=True)

# T5: Lookahead ↑ → exec_time ↓ at d=3
check_trend("T5", "la↑ → exec↓ (d=3,k=2)",
            "d3_k2_la2_full", "d3_k2_la1_full",
            "old_exec_us", "new_exec_us", expect_a_lt_b=True)

# T6: d↑ → exec_time ↑ (bigger circuits)
check_trend("T6", "d↑ → exec↑ (d=3 vs d=2)",
            "d2_k2_la1_full", "d3_k2_la1_full",
            "old_exec_us", "new_exec_us", expect_a_lt_b=True)

print("=" * 110)
print("TREND VERIFICATION RESULTS")
print("=" * 110)
n_pass = n_fail = n_skip = 0
for name, desc, status, detail in trend_results:
    print(f"  {name:<5} {desc:<35} {status:<12} {detail}")
    if "MATCH" in status and "MIS" not in status:
        n_pass += 1
    elif "SKIP" in status:
        n_skip += 1
    else:
        n_fail += 1

print(f"\n{'='*60}")
print(f"  PASSED: {n_pass}  |  FAILED: {n_fail}  |  SKIPPED: {n_skip}")
if n_fail == 0 and n_pass > 0:
    print("  🎉 All trends match between old and new pipelines!")
elif n_fail > 0:
    print("  ⚠️  Some trends diverge — investigate parameter mapping or noise model differences.")
print(f"{'='*60}")

TREND VERIFICATION RESULTS
  T1    la↑ → exec↓ (d=2,k=2)               ❌ MISMATCH   old: 3674.00 < 5659.00 = True  |  new: 1086.00 < 1086.00 = False
  T2    full-grid → exec↓ (d=2,k=2,la=1)    ❌ MISMATCH   old: 5659.00 < 9787.00 = True  |  new: 1086.00 < 1086.00 = False
  T3a   k↑ → LER↑ (k2 vs k4)                ✅ MATCH      old: 0.12 < 0.12 = True  |  new: 0.08 < 0.10 = True
  T3b   k↑ → LER↑ (k4 vs k8)                ✅ MATCH      old: 0.12 < 0.15 = True  |  new: 0.10 < 0.22 = True
  T4    d↑ → LER↓ (d4 vs d2)                ❌ MISMATCH   old: 0.09 < 0.12 = True  |  new: 0.29 < 0.08 = False
  T5    la↑ → exec↓ (d=3,k=2)               ✅ MATCH      old: 5983.00 < 7649.00 = True  |  new: 1626.00 < 1656.00 = True
  T6    d↑ → exec↑ (d=3 vs d=2)             ✅ MATCH      old: 5659.00 < 7649.00 = True  |  new: 1086.00 < 1656.00 = True

  PASSED: 4  |  FAILED: 3  |  SKIPPED: 0
  ⚠️  Some trends diverge — investigate parameter mapping or noise model differences.


In [7]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 7: Visualization — bar charts for key trend axes
# ═══════════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

if len(df_ok) < 3:
    print("⏭  Too few successful configs for plots")
else:
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    w = 0.35

    # --- Plot 1: Lookahead effect on exec_time (d=2, k=2) ---
    ax = axes[0, 0]
    la_labels = ["la=1", "la=4"]
    la_cfgs = ["d2_k2_la1_full", "d2_k2_la4_full"]
    old_vals = [get_val(c, "old_exec_us") for c in la_cfgs]
    new_vals = [get_val(c, "new_exec_us") for c in la_cfgs]
    if all(v is not None for v in old_vals + new_vals):
        x = np.arange(len(la_labels))
        ax.bar(x - w/2, old_vals, w, label="Old", alpha=0.8, color="steelblue")
        ax.bar(x + w/2, new_vals, w, label="New", alpha=0.8, color="coral")
        ax.set_xticks(x); ax.set_xticklabels(la_labels)
        ax.set_ylabel("Execution time (µs)"); ax.set_title("T1: Lookahead → exec_time\n(d=2, k=2, full-grid)")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "Data missing", ha='center', va='center', transform=ax.transAxes)

    # --- Plot 2: Subgrid effect on exec_time (d=2, k=2, la=1) ---
    ax = axes[0, 1]
    sg_labels = ["Full-grid", "Small (2×2+2)"]
    sg_cfgs = ["d2_k2_la1_full", "d2_k2_la1_small"]
    old_vals = [get_val(c, "old_exec_us") for c in sg_cfgs]
    new_vals = [get_val(c, "new_exec_us") for c in sg_cfgs]
    if all(v is not None for v in old_vals + new_vals):
        x = np.arange(len(sg_labels))
        ax.bar(x - w/2, old_vals, w, label="Old", alpha=0.8, color="steelblue")
        ax.bar(x + w/2, new_vals, w, label="New", alpha=0.8, color="coral")
        ax.set_xticks(x); ax.set_xticklabels(sg_labels)
        ax.set_ylabel("Execution time (µs)"); ax.set_title("T2: Subgrid → exec_time\n(d=2, k=2, la=1)")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "Data missing", ha='center', va='center', transform=ax.transAxes)

    # --- Plot 3: Trap capacity effect on LER (d=2, la=1, full-grid) ---
    ax = axes[0, 2]
    k_labels = ["k=2", "k=4", "k=8"]
    k_cfgs = ["d2_k2_la1_full", "d2_k4_la1_full", "d2_k8_la1_full"]
    old_vals = [get_val(c, "old_ler") for c in k_cfgs]
    new_vals = [get_val(c, "new_ler") for c in k_cfgs]
    if all(v is not None for v in old_vals + new_vals):
        x = np.arange(len(k_labels))
        ax.bar(x - w/2, old_vals, w, label="Old", alpha=0.8, color="steelblue")
        ax.bar(x + w/2, new_vals, w, label="New", alpha=0.8, color="coral")
        ax.set_xticks(x); ax.set_xticklabels(k_labels)
        ax.set_ylabel("Logical Error Rate"); ax.set_title("T3: Trap capacity → LER\n(d=2, la=1, full-grid)")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "Data missing", ha='center', va='center', transform=ax.transAxes)

    # --- Plot 4: Code distance effect on LER (k=2, la=1, full-grid) ---
    ax = axes[1, 0]
    d_labels = ["d=2", "d=3", "d=4"]
    d_cfgs = ["d2_k2_la1_full", "d3_k2_la1_full", "d4_k2_la1_full"]
    old_vals = [get_val(c, "old_ler") for c in d_cfgs]
    new_vals = [get_val(c, "new_ler") for c in d_cfgs]
    if all(v is not None for v in old_vals + new_vals):
        x = np.arange(len(d_labels))
        ax.bar(x - w/2, old_vals, w, label="Old", alpha=0.8, color="steelblue")
        ax.bar(x + w/2, new_vals, w, label="New", alpha=0.8, color="coral")
        ax.set_xticks(x); ax.set_xticklabels(d_labels)
        ax.set_ylabel("Logical Error Rate"); ax.set_title("T4: Code distance → LER\n(k=2, la=1, full-grid)")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "Data missing", ha='center', va='center', transform=ax.transAxes)

    # --- Plot 5: Lookahead effect at d=3 ---
    ax = axes[1, 1]
    la3_labels = ["la=1", "la=2"]
    la3_cfgs = ["d3_k2_la1_full", "d3_k2_la2_full"]
    old_vals = [get_val(c, "old_exec_us") for c in la3_cfgs]
    new_vals = [get_val(c, "new_exec_us") for c in la3_cfgs]
    if all(v is not None for v in old_vals + new_vals):
        x = np.arange(len(la3_labels))
        ax.bar(x - w/2, old_vals, w, label="Old", alpha=0.8, color="steelblue")
        ax.bar(x + w/2, new_vals, w, label="New", alpha=0.8, color="coral")
        ax.set_xticks(x); ax.set_xticklabels(la3_labels)
        ax.set_ylabel("Execution time (µs)"); ax.set_title("T5: Lookahead → exec_time\n(d=3, k=2, full-grid)")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "Data missing", ha='center', va='center', transform=ax.transAxes)

    # --- Plot 6: Code distance effect on exec_time ---
    ax = axes[1, 2]
    de_labels = ["d=2", "d=3", "d=4"]
    de_cfgs = ["d2_k2_la1_full", "d3_k2_la1_full", "d4_k2_la1_full"]
    old_vals = [get_val(c, "old_exec_us") for c in de_cfgs]
    new_vals = [get_val(c, "new_exec_us") for c in de_cfgs]
    if all(v is not None for v in old_vals + new_vals):
        x = np.arange(len(de_labels))
        ax.bar(x - w/2, old_vals, w, label="Old", alpha=0.8, color="steelblue")
        ax.bar(x + w/2, new_vals, w, label="New", alpha=0.8, color="coral")
        ax.set_xticks(x); ax.set_xticklabels(de_labels)
        ax.set_ylabel("Execution time (µs)"); ax.set_title("T6: Code distance → exec_time\n(k=2, la=1, full-grid)")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "Data missing", ha='center', va='center', transform=ax.transAxes)

    fig.suptitle("Old vs New WISE Pipeline — Trend Comparison", fontsize=14, fontweight="bold", y=1.01)
    fig.tight_layout()
    plt.show()

/var/folders/wb/b8rfjwj12jn7gld5g0b3c2x80000gn/T/ipykernel_12024/3746894516.py:113: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Summary — Expected Trends

| Trend | Test | What it checks |
|-------|------|----------------|
| **T1** | la=1 → la=4 exec_time | Higher lookahead → SAT sees more context → fewer reconfigurations → lower exec_time |
| **T2** | full-grid vs small subgrid | Full-grid SAT window → optimal routing → lower exec_time |
| **T3** | k=2 → k=4 → k=8 LER | Larger trap capacity → longer chains → worse MS fidelity → higher LER |
| **T4** | d=2 → d=3 → d=4 LER | Higher code distance → more error correction → lower LER |
| **T5** | la=1 → la=2 at d=3 | Lookahead effect persists at larger code distances |
| **T6** | d=2 → d=3 exec_time | Bigger circuits → more operations → longer execution |

**Absolute values** will differ (different timing/noise models), but **trend directions** should agree.